# PIPELINE (Fluxo para criação e avaliação de algoritmos de Machine Learning)

Sequência de etapas de processamento, onde cada etapa executa uma transformação nos dados, e a última etapa é geralmente o modelo de machine learning

## DATA LEAKAGE (VAZAMENTO DE DADOS)

Quando informações do conjunto de teste são utilizadas, direta ou indiretamente, durante o treinamento do modelo, **pode** ocorrer o vazamento de dados, isto é, o modelo vê informações que não teria na "vida real", **pode** gerar resultados ilusoriamente bons.

Recomenda-se separar em treino e teste antes do pré-processamento dos dados (escalonamento, substituição ou inclusão de valores, variáveis dummies, transformação de variáveis categóricas para numéricas, redução de dimensionalidade e balanceamento).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
# Carregar dataset
df = pd.read_csv('data/heart_tratado.csv', sep=';', encoding='utf-8')

In [4]:
# Separar previsores e alvo
X = df.drop('HeartDisease', axis=1) # previsores
y = df['HeartDisease'] # alvo

In [5]:
# Separar treino e teste antes de qualquer transformação para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [6]:
# Identificar colunas categóricas e numéricas
categoricas = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
numericas = [col for col in X.columns if col not in categoricas]

In [7]:
# Pipeline com OneHotEncoder
preprocess_onehot = ColumnTransformer([
    ('num', Pipeline([
        # ('imp', SimpleImputer(strategy='mean')),
        # substituir valores missing pela média
        ('scaler', StandardScaler()) # Padronização
        # ('scaler', MinMaxScaler()) # Normalização
    ]), numericas),

    ('cat', Pipeline([
        # ('imp', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categoricas)
])

# Pipeline com OrdinalEncoder
# OrdinalEncoder é similar ao LabelEnconder, porém tem as vantagens de
# atuar num conjunto de atributos, ao invés de atuação individual e
# é recomendado para uso em atributos previsores ou variáveis independentes.
preprocess_ordinal = ColumnTransformer([
    ('num', Pipeline([
        # ('imp', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()) # Padronização
        # ('scaler', MinMaxScaler()) # Normalização
    ]), numericas),

    ('cat', Pipeline([
        # ('imp', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder())
    ]), categoricas)
])

In [8]:
# Treino de modelo com OnehotEncoder
pipeline_onehot = Pipeline([
    ('pre', preprocess_onehot),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

In [9]:
# Treino de modelo com OrdinalEncoder
pipeline_ordinal = Pipeline([
    ('pre', preprocess_ordinal),
    ('clf', XGBClassifier(max_depth=2, learning_rate=0.05, n_estimators=250, random_state=3))
])

In [10]:
# Avaliação em dados de teste
print("\nAvaliação com dados de teste (OneHot):")
pipeline_onehot.fit(X_train, y_train)
y_pred1 = pipeline_onehot.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred1):.4f}")
print(confusion_matrix(y_test, y_pred1))
print(classification_report(y_test, y_pred1))

print("\nAvaliação com dados de teste (Ordinal):")
pipeline_ordinal.fit(X_train, y_train)
y_pred2 = pipeline_ordinal.predict(X_test)
print(f"Acurácia: {accuracy_score(y_test, y_pred2):.4f}")
print(confusion_matrix(y_test, y_pred2))
print(classification_report(y_test, y_pred2))


Avaliação com dados de teste (OneHot):
Acurácia: 0.8659
[[102  19]
 [ 18 137]]
              precision    recall  f1-score   support

           0       0.85      0.84      0.85       121
           1       0.88      0.88      0.88       155

    accuracy                           0.87       276
   macro avg       0.86      0.86      0.86       276
weighted avg       0.87      0.87      0.87       276


Avaliação com dados de teste (Ordinal):
Acurácia: 0.8587
[[101  20]
 [ 19 136]]
              precision    recall  f1-score   support

           0       0.84      0.83      0.84       121
           1       0.87      0.88      0.87       155

    accuracy                           0.86       276
   macro avg       0.86      0.86      0.86       276
weighted avg       0.86      0.86      0.86       276



In [11]:
# Comparação via validação cruzada
print("OneHot:")
scores1 = cross_val_score(pipeline_onehot, X, y, cv=5)
print(f"Acurácia média: {scores1.mean():.4f}")

print("\nOrdinal:")
scores2 = cross_val_score(pipeline_ordinal, X, y, cv=5)
print(f"Acurácia média: {scores2.mean():.4f}")

OneHot:
Acurácia média: 0.8331

Ordinal:
Acurácia média: 0.8364
